# Optimizing the ANN

## Solutions

- Adding More Data
- Reducing complexity
- Regularization
- Dropout
- Data Augmentation (CNN)
- Batch Normalization
- Early Stopping

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

In [2]:
torch.manual_seed(42)

In [3]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu') 
print(f"Using device: {device}")

Using device: mps


In [5]:
df = pd.read_csv('data/fashion-mnist_train.csv')
df.head(2)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [7]:
#train test split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [8]:
#scaling the values
X_train = X_train/255.0
X_test = X_test/255.0

In [9]:
#Create custom dataset

class CustomDataset(Dataset):
    def __init__(self,features,label):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.label = torch.tensor(label, dtype=torch.long)

    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, index):
        return self.features[index], self.label[index]

In [10]:
#create train dataset
train_dataset = CustomDataset(X_train, y_train)

In [11]:
#create test dataset
test_dataset = CustomDataset(X_test,y_test)

In [12]:
#create train and test loader

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, pin_memory=True)

In [14]:
#define nn class
class Mynn(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(64,10)
        )

    def forward(self,x):
        return self.model(x)

In [15]:
#set learning rate and epochs
epochs = 100
learning_rate = 0.1

In [16]:
#instantiate the model
model = Mynn(X_train.shape[1])
model = model.to(device)
#loss function 
criterion = nn.CrossEntropyLoss()

#optimizer
optimizer = optim.SGD(model.parameters(), learning_rate, weight_decay=1e-4)

In [17]:
#training loop

for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features, batch_label in train_loader:
        batch_features, batch_label = batch_features.to(device), batch_label.to(device)
        #forwardpass
        outputs = model(batch_features)
        #calculate loss
        loss = criterion(outputs, batch_label)
        #backpass
        optimizer.zero_grad()
        loss.backward()
        #updateparams
        optimizer.step()

        total_epoch_loss = total_epoch_loss + loss.item()

    avg_loss = total_epoch_loss/len(train_loader)
    print(f'Epoch: {epoch+1}, Loss:{avg_loss:.4f}')
    

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch: 1, Loss:0.6591
Epoch: 2, Loss:0.4820
Epoch: 3, Loss:0.4397
Epoch: 4, Loss:0.4164
Epoch: 5, Loss:0.3984
Epoch: 6, Loss:0.3828
Epoch: 7, Loss:0.3726
Epoch: 8, Loss:0.3625
Epoch: 9, Loss:0.3522
Epoch: 10, Loss:0.3456
Epoch: 11, Loss:0.3376
Epoch: 12, Loss:0.3312
Epoch: 13, Loss:0.3237
Epoch: 14, Loss:0.3189
Epoch: 15, Loss:0.3101
Epoch: 16, Loss:0.3081
Epoch: 17, Loss:0.3051
Epoch: 18, Loss:0.3018
Epoch: 19, Loss:0.2951
Epoch: 20, Loss:0.2909
Epoch: 21, Loss:0.2855
Epoch: 22, Loss:0.2784
Epoch: 23, Loss:0.2770
Epoch: 24, Loss:0.2762
Epoch: 25, Loss:0.2737
Epoch: 26, Loss:0.2701
Epoch: 27, Loss:0.2632
Epoch: 28, Loss:0.2641
Epoch: 29, Loss:0.2608
Epoch: 30, Loss:0.2573
Epoch: 31, Loss:0.2559
Epoch: 32, Loss:0.2531
Epoch: 33, Loss:0.2502
Epoch: 34, Loss:0.2473
Epoch: 35, Loss:0.2480
Epoch: 36, Loss:0.2463
Epoch: 37, Loss:0.2427
Epoch: 38, Loss:0.2376
Epoch: 39, Loss:0.2367
Epoch: 40, Loss:0.2371
Epoch: 41, Loss:0.2336
Epoch: 42, Loss:0.2320
Epoch: 43, Loss:0.2305
Epoch: 44, Loss:0.23

In [18]:
#set model to eval mode

model.eval()

Mynn(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [19]:
#evaluation code

total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_label in test_loader:
        batch_features, batch_label = batch_features.to(device), batch_label.to(device)
        output = model(batch_features)
        _, predicted = torch.max(output,1)
        
        total = total + batch_label.shape[0]
        correct = correct + (predicted == batch_label).sum().item()
    
print(correct/total)


0.8906666666666667


In [20]:
#evaluation code

total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_label in train_loader:
        batch_features, batch_label = batch_features.to(device), batch_label.to(device)
        output = model(batch_features)
        _, predicted = torch.max(output,1)
        
        total = total + batch_label.shape[0]
        correct = correct + (predicted == batch_label).sum().item()
    
print(correct/total)


0.962625
